# Notebook 6 — Analyse statistique des extrêmes
## Ajustement de lois (GEV, Gumbel, Lognormale) sur les maxima annuels

---

## Objectif

Ce notebook ajuste des **lois statistiques extrêmes** sur les maxima annuels
d'intensité et calcule les **valeurs de retour** pour T = 2, 5, 10, 20, 50, 100 ans.

### Théorie des valeurs extrêmes

La théorie des valeurs extrêmes (EVT) stipule que les maxima annuels d'une variable
suivent asymptotiquement une loi **GEV (Gumbel, Fréchet ou Weibull)** :

$$F(x) = \exp\left[-\left(1 + \xi \frac{x - \mu}{\sigma}\right)^{-1/\xi}\right]$$

où $\mu$ = localisation, $\sigma$ = échelle, $\xi$ = forme.

| $\xi$ | Loi | Description |
|-------|-----|-------------|
| $\xi = 0$ | **Gumbel** | Cas le plus fréquent en hydrologie |
| $\xi > 0$ | **Fréchet** | Queue lourde (extrêmes très élevés) |
| $\xi < 0$ | **Weibull** | Borne supérieure finie |

### Améliorations apportées dans cette version

1. **Test de tendance Mann-Kendall** — vérifie l'hypothèse de stationnarité
2. **Tests d'adéquation formels** — Kolmogorov-Smirnov et Anderson-Darling
3. **Discussion du paramètre de forme** de la GEV

### Livrables

| Fichier | Contenu |
|---------|---------|
| `figures/n6/fig1_lois_ajustees.png` | PDF des 3 lois sur l'histogramme |
| `figures/n6/fig2_qqplot.png` | Q-Q plots (positions de Gringorten) |
| `figures/n6/fig3_periodes_retour.png` | Courbes de valeurs de retour |
| `figures/n6/fig4_intervalles_confiance.png` | IC 95% Bootstrap |
| `figures/n6/fig5_stationnarite.png` | Test de stationnarité des maxima |

---
**Auteur :** AY2K
**Encadreur :** Prof. Titembaye Donald
**Établissement :** École Polytechnique de Lomé (EPL)
**Projet :** Modélisation des courbes IDF — Station de Kara, Nord Togo


## 0. Configuration et imports


In [ ]:
import os
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning, module='scipy')

NB      = 'n6'
FIG_DIR = f'figures/{NB}'
os.makedirs(FIG_DIR, exist_ok=True)

print(f'Répertoire figures : {FIG_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from scipy.stats import kstest, anderson

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.6,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'figure.dpi':       100,
})

print('Imports chargés')

---
## Section 1 — Chargement et extraction des maxima annuels

On charge la série complète produite par n5 et on extrait le **maximum annuel**
d'intensité. Ces maxima annuels sont la base de l'analyse des extrêmes.


In [ ]:
df = pd.read_csv('data/serie_complete.csv', parse_dates=['date'])
print(f'Série complète : {len(df)} lignes | {df.date.min().date()} → {df.date.max().date()}')

df['annee'] = df['date'].dt.year

# Maxima annuels d'intensité (mm/h)
maxima = df.groupby('annee')['intensite_max_mm_h'].max().dropna().reset_index()
maxima.columns = ['annee', 'imax_mm_h']
maxima = maxima.sort_values('annee').reset_index(drop=True)

# Informer sur la source (observé vs reconstitué) de chaque maximum
def source_du_max(annee, df_serie):
    yr = df_serie[df_serie['annee'] == annee]
    idx_max = yr['intensite_max_mm_h'].idxmax()
    return yr.loc[idx_max, 'source']

maxima['source'] = maxima['annee'].apply(lambda a: source_du_max(a, df))

print(f'\nMaxima annuels : {len(maxima)} années')
print(f'Observés       : {(maxima.source == "observé").sum()} années')
print(f'Reconstitués   : {(maxima.source == "reconstitué").sum()} années')
print(f'Min    : {maxima.imax_mm_h.min():.1f} mm/h')
print(f'Max    : {maxima.imax_mm_h.max():.1f} mm/h')
print(f'Moyenne: {maxima.imax_mm_h.mean():.1f} mm/h')
print(f'Écart-type: {maxima.imax_mm_h.std():.1f} mm/h')
print()
print(maxima.to_string(index=False))

---
## Section 2 — Test de stationnarité (Mann-Kendall)

Avant d'ajuster une loi statistique, on vérifie l'hypothèse de **stationnarité** :
la série de maxima ne doit pas présenter de tendance temporelle significative.

### Test de Mann-Kendall

Le test de Mann-Kendall est un test non-paramétrique de tendance monotone.
- **H0 (hypothèse nulle)** : pas de tendance dans la série
- **H1** : il existe une tendance monotone (croissante ou décroissante)
- **Seuil de rejet** : p-value < 0.05 → tendance significative

On calcule également la **pente de Sen** (estimateur robuste de la pente).


In [ ]:
# ==============================================================================
# TEST DE MANN-KENDALL (implémentation manuelle, sans bibliothèque externe)
# ==============================================================================

def mann_kendall(x):
    """
    Test de Mann-Kendall pour une tendance monotone.
    Retourne la statistique S, la variance Var(S), et la p-value.
    """
    n = len(x)
    S = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            diff = x[j] - x[i]
            if diff > 0:
                S += 1
            elif diff < 0:
                S -= 1
    # Variance de S (sans égalités)
    var_S = n * (n - 1) * (2 * n + 5) / 18
    # Statistique Z standardisée
    if S > 0:
        Z = (S - 1) / np.sqrt(var_S)
    elif S < 0:
        Z = (S + 1) / np.sqrt(var_S)
    else:
        Z = 0.0
    p_value = 2 * (1 - sp_stats.norm.cdf(abs(Z)))
    return S, Z, p_value


def pente_sen(x, y):
    """Estimateur de pente de Sen (médiane des pentes deux à deux)."""
    pentes = []
    n = len(x)
    for i in range(n - 1):
        for j in range(i + 1, n):
            if x[j] != x[i]:
                pentes.append((y[j] - y[i]) / (x[j] - x[i]))
    return np.median(pentes)


data_mk = maxima['imax_mm_h'].values
annees  = maxima['annee'].values

S, Z_mk, p_mk = mann_kendall(data_mk)
pente = pente_sen(annees.astype(float), data_mk)

print('=== Test de Mann-Kendall ===')
print(f'Statistique S  : {S}')
print(f'Statistique Z  : {Z_mk:.3f}')
print(f'p-value        : {p_mk:.4f}')
print(f'Pente de Sen   : {pente:.2f} mm/h par année')
print()
if p_mk < 0.05:
    print('RÉSULTAT : Tendance SIGNIFICATIVE (p < 0.05)')
    print('  → Hypothèse de stationnarité REJETÉE')
    print('  → Les résultats IDF doivent être interprétés avec précaution.')
else:
    print('RÉSULTAT : Pas de tendance significative (p ≥ 0.05)')
    print('  → Hypothèse de stationnarité ACCEPTÉE')
    print('  → L\'ajustement de lois stationnaires est justifié.')

---
## Section 3 — Ajustement des lois statistiques

On teste trois lois et on retient celle avec l'**AIC minimal** :
- **GEV** (3 paramètres) — loi générale, englobe Gumbel et Fréchet
- **Gumbel** (2 paramètres) — cas particulier GEV, standard OMM
- **Log-normale** (2 paramètres) — alternative classique

$$\text{AIC} = -2 \ln(L) + 2k$$


In [ ]:
data = maxima['imax_mm_h'].values
n    = len(data)


def aic_bic(log_lik, k, n):
    """AIC et BIC à partir de la log-vraisemblance."""
    aic = -2 * log_lik + 2 * k
    bic = -2 * log_lik + k * np.log(n)
    return aic, bic


resultats_lois = []

# --- GEV (3 paramètres) ---
try:
    gev_params = sp_stats.genextreme.fit(data)
    gev_ll     = np.sum(sp_stats.genextreme.logpdf(data, *gev_params))
    gev_aic, gev_bic = aic_bic(gev_ll, 3, n)
    resultats_lois.append({'Loi': 'GEV', 'Paramètres': gev_params,
                            'AIC': gev_aic, 'BIC': gev_bic,
                            'dist': sp_stats.genextreme})
    xi = gev_params[0]
    print(f'GEV    : shape(ξ)={xi:.3f}, loc={gev_params[1]:.2f}, '
          f'scale={gev_params[2]:.2f} | AIC={gev_aic:.1f}')
    if abs(xi) > 0.5:
        print(f'  Attention : ξ={xi:.3f} est élevé (|ξ| > 0.5 est inhabituel en hydrologie)')
except Exception as e:
    print(f'GEV non convergée : {e}')

# --- Gumbel (2 paramètres) ---
try:
    gum_params = sp_stats.gumbel_r.fit(data)
    gum_ll     = np.sum(sp_stats.gumbel_r.logpdf(data, *gum_params))
    gum_aic, gum_bic = aic_bic(gum_ll, 2, n)
    resultats_lois.append({'Loi': 'Gumbel', 'Paramètres': gum_params,
                            'AIC': gum_aic, 'BIC': gum_bic,
                            'dist': sp_stats.gumbel_r})
    print(f'Gumbel : loc={gum_params[0]:.2f}, scale={gum_params[1]:.2f} | AIC={gum_aic:.1f}')
except Exception as e:
    print(f'Gumbel non ajustée : {e}')

# --- Lognormale (2 paramètres, loc=0 fixé) ---
try:
    ln_params = sp_stats.lognorm.fit(data, floc=0)
    ln_ll     = np.sum(sp_stats.lognorm.logpdf(data, *ln_params))
    ln_aic, ln_bic = aic_bic(ln_ll, 2, n)
    resultats_lois.append({'Loi': 'Lognormale', 'Paramètres': ln_params,
                            'AIC': ln_aic, 'BIC': ln_bic,
                            'dist': sp_stats.lognorm})
    print(f'Lognorm: shape={ln_params[0]:.3f}, scale={ln_params[2]:.2f} | AIC={ln_aic:.1f}')
except Exception as e:
    print(f'Lognormale non ajustée : {e}')

# Sélection par AIC
df_lois  = pd.DataFrame([{k: v for k, v in r.items() if k != 'dist'}
                          for r in resultats_lois])
df_lois  = df_lois.sort_values('AIC').reset_index(drop=True)
best_loi = df_lois.iloc[0]['Loi']
best_loi_dict = next(r for r in resultats_lois if r['Loi'] == best_loi)

print(f'\n=== Classement AIC ===')
print(df_lois[['Loi', 'AIC', 'BIC']].to_string(index=False))
print(f'\n→ Loi retenue : {best_loi} (AIC minimal)')

---
## Section 4 — Tests d'adéquation formels

### Pourquoi tester l'adéquation ?

Le critère AIC compare les lois **entre elles**, mais ne garantit pas qu'aucune d'elles
ne soit acceptable. Un test d'adéquation vérifie si une loi donnée est **compatible**
avec les données.

### Test de Kolmogorov-Smirnov (KS)

Compare la FDR empirique à la FDR théorique.
- p > 0.05 : la loi est compatible avec les données (on ne rejette pas H0)
- p < 0.05 : la loi est rejetée

### Test d'Anderson-Darling (AD)

Plus sensible aux queues de distribution que le KS — particulièrement adapté
aux extrêmes hydrologiques. On compare la statistique A² aux valeurs critiques
pour les niveaux de signification 15%, 10%, 5%, 2.5%, 1%.


In [ ]:
# ==============================================================================
# TESTS D'ADÉQUATION : KS et Anderson-Darling
# ==============================================================================

print('=' * 60)
print('TESTS D\'ADÉQUATION')
print('=' * 60)

for r in resultats_lois:
    nom    = r['Loi']
    params = r['Paramètres']
    dist   = r['dist']

    print(f'\n--- {nom} ---')

    # --- Test KS ---
    try:
        ks_stat, ks_pval = kstest(data, lambda x: dist.cdf(x, *params))
        verdict_ks = 'ACCEPTABLE (p ≥ 0.05)' if ks_pval >= 0.05 else 'REJETÉ (p < 0.05)'
        print(f'  KS  : stat={ks_stat:.4f}, p-value={ks_pval:.4f}  → {verdict_ks}')
    except Exception as e:
        print(f'  KS  : erreur ({e})')

    # --- Test Anderson-Darling (via simulation bootstrap) ---
    # scipy.stats.anderson ne supporte pas directement toutes les distributions.
    # On utilise une approximation par simulation de Monte Carlo.
    try:
        N_SIM  = 1000
        np.random.seed(42)
        ad_obs = _anderson_darling_stat(data, dist, params)

        # Distribution de référence par simulation sous H0
        ad_sim = []
        for _ in range(N_SIM):
            sample_sim = dist.rvs(*params, size=len(data), random_state=None)
            try:
                params_sim = dist.fit(sample_sim)
                ad_sim.append(_anderson_darling_stat(sample_sim, dist, params_sim))
            except Exception:
                pass

        if len(ad_sim) > 0:
            p_ad = np.mean(np.array(ad_sim) >= ad_obs)
            verdict_ad = 'ACCEPTABLE (p ≥ 0.05)' if p_ad >= 0.05 else 'REJETÉ (p < 0.05)'
            print(f'  AD  : stat={ad_obs:.4f}, p-value≈{p_ad:.4f} (Monte Carlo, n={N_SIM})  → {verdict_ad}')
    except Exception as e:
        print(f'  AD  : erreur ({e})')

print('\n' + '=' * 60)
print('INTERPRÉTATION')
print('=' * 60)
print('Un test acceptable (p ≥ 0.05) signifie que la loi n\'est pas rejetée.')
print('La loi retenue par AIC est utilisée si elle passe au moins le test KS.')

In [ ]:
# Fonction AD (définie avant utilisation)
def _anderson_darling_stat(x, dist, params):
    """Calcule la statistique A² de Anderson-Darling."""
    x_sorted = np.sort(x)
    n = len(x_sorted)
    cdf_vals = dist.cdf(x_sorted, *params)
    # Clip pour éviter log(0)
    cdf_vals = np.clip(cdf_vals, 1e-10, 1 - 1e-10)
    i = np.arange(1, n + 1)
    S = np.sum((2 * i - 1) * (np.log(cdf_vals) + np.log(1 - cdf_vals[::-1])))
    A2 = -n - S / n
    return A2

# Relancer les tests avec la fonction définie
print('=' * 60)
print('TESTS D\'ADÉQUATION (résultats complets)')
print('=' * 60)

resultats_tests = []
for r in resultats_lois:
    nom, params, dist = r['Loi'], r['Paramètres'], r['dist']
    row = {'Loi': nom}
    # KS
    try:
        ks_stat, ks_pval = kstest(data, lambda x, p=params, d=dist: d.cdf(x, *p))
        row['KS stat']   = round(ks_stat, 4)
        row['KS p-value'] = round(ks_pval, 4)
        row['KS verdict'] = 'OK' if ks_pval >= 0.05 else 'Rejeté'
    except:
        row['KS stat'] = row['KS p-value'] = row['KS verdict'] = 'Erreur'
    # AD
    try:
        ad_obs = _anderson_darling_stat(data, dist, params)
        row['AD stat'] = round(ad_obs, 4)
    except:
        row['AD stat'] = 'Erreur'
    resultats_tests.append(row)

df_tests = pd.DataFrame(resultats_tests)
print(df_tests.to_string(index=False))
print()
print('Rappel : KS verdict "OK" signifie p ≥ 0.05 → loi non rejetée.')

---
## Section 5 — Valeurs de retour

La **valeur de retour** $x_T$ est l'intensité dépassée en moyenne une fois tous les $T$ ans :

$$x_T = F^{-1}\left(1 - \frac{1}{T}\right)$$


In [ ]:
periodes          = [2, 5, 10, 20, 50, 100]
proba_depassement = [1 - 1/T for T in periodes]

tableau_retour = {'Période (ans)': periodes}
for r in resultats_lois:
    try:
        vals = [r['dist'].ppf(p, *r['Paramètres']) for p in proba_depassement]
        tableau_retour[r['Loi']] = [round(v, 1) for v in vals]
    except Exception as e:
        tableau_retour[r['Loi']] = ['N/A'] * len(periodes)

df_retour = pd.DataFrame(tableau_retour)
print('=== Valeurs de retour (mm/h) — Intensité max annuelle ===')
print(df_retour.to_string(index=False))
print()
print('Note : ces valeurs concernent l\'intensité maximale annuelle (toutes durées).')
print('Les valeurs de retour par durée sont dans le Notebook 7.')

---
## Section 6 — Figures


In [ ]:
# ==============================================================================
# FIGURE 1 — Ajustement des lois sur les maxima annuels
# ==============================================================================
x_range     = np.linspace(data.min() * 0.8, data.max() * 1.3, 500)
colors_lois = {'GEV': '#1565C0', 'Gumbel': '#E53935', 'Lognormale': '#2E7D32'}

fig, ax = plt.subplots(figsize=(11, 6))
ax.hist(data, bins=12, density=True, alpha=0.45, color='#B0BEC5',
        edgecolor='white', label='Données observées')

for r in resultats_lois:
    try:
        pdf_vals = r['dist'].pdf(x_range, *r['Paramètres'])
        style = '-' if r['Loi'] == best_loi else '--'
        lw    = 2.5 if r['Loi'] == best_loi else 1.5
        ax.plot(x_range, pdf_vals, style,
                color=colors_lois.get(r['Loi'], 'black'),
                linewidth=lw, label=f"{r['Loi']} (AIC={r['AIC']:.1f})")
    except Exception:
        pass

ax.set_xlabel('Intensité max annuelle (mm/h)')
ax.set_ylabel('Densité')
ax.set_title('Ajustement des lois aux maxima annuels — Kara',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig1_lois_ajustees.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig1 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 2 — Q-Q plots (positions de Gringorten)
# ==============================================================================
n_lois    = len(resultats_lois)
fig, axes = plt.subplots(1, n_lois, figsize=(5 * n_lois, 5))
if n_lois == 1:
    axes = [axes]

data_sorted = np.sort(data)
pp          = (np.arange(1, n + 1) - 0.44) / (n + 0.12)   # positions de Gringorten

for ax, r in zip(axes, resultats_lois):
    try:
        theoriques = r['dist'].ppf(pp, *r['Paramètres'])
        ax.scatter(theoriques, data_sorted, s=35, alpha=0.75,
                   color=colors_lois.get(r['Loi'], 'steelblue'))
        lim_min = min(theoriques.min(), data_sorted.min()) * 0.9
        lim_max = max(theoriques.max(), data_sorted.max()) * 1.1
        ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=1.5)
        ax.set_xlabel('Quantiles théoriques (mm/h)')
        ax.set_ylabel('Quantiles observés (mm/h)')
        ax.set_title(f'{r["Loi"]}', fontweight='bold')
    except Exception as e:
        ax.set_title(f'{r["Loi"]} (erreur : {e})')

plt.suptitle('Q-Q plots — Lois ajustées (positions de Gringorten)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig2_qqplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig2 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 3 — Courbes de valeurs de retour
# ==============================================================================
T_range = np.logspace(np.log10(1.1), np.log10(200), 200)
p_range = 1 - 1 / T_range

fig, ax = plt.subplots(figsize=(12, 6))
for r in resultats_lois:
    try:
        vr    = [r['dist'].ppf(p, *r['Paramètres']) for p in p_range]
        style = '-' if r['Loi'] == best_loi else '--'
        lw    = 2.5 if r['Loi'] == best_loi else 1.5
        ax.plot(T_range, vr, style, color=colors_lois.get(r['Loi'], 'black'),
                linewidth=lw, label=r['Loi'])
    except Exception:
        pass

# Points observés (Gringorten)
T_obs = 1 / (1 - pp)
ax.scatter(T_obs, data_sorted, zorder=5, color='black', s=40,
           label='Maxima observés')

for T in [10, 20, 50, 100]:
    ax.axvline(T, color='grey', linestyle=':', alpha=0.4)

ax.set_xscale('log')
ax.set_xlabel('Période de retour T (ans)')
ax.set_ylabel('Intensité max (mm/h)')
ax.set_title('Valeurs de retour — Station Kara',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig3_periodes_retour.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig3 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 4 — Intervalles de confiance 95% (Bootstrap)
# ==============================================================================
N_BOOTSTRAP = 500
best_r      = best_loi_dict
vr_bootstrap = np.zeros((N_BOOTSTRAP, len(T_range)))

np.random.seed(42)
for i in range(N_BOOTSTRAP):
    sample = np.random.choice(data, size=len(data), replace=True)
    try:
        params_b     = best_r['dist'].fit(sample)
        vr_bootstrap[i] = [best_r['dist'].ppf(p, *params_b) for p in p_range]
    except Exception:
        vr_bootstrap[i] = np.nan

vr_median = np.nanpercentile(vr_bootstrap, 50,  axis=0)
vr_lower  = np.nanpercentile(vr_bootstrap,  2.5, axis=0)
vr_upper  = np.nanpercentile(vr_bootstrap, 97.5, axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(T_range, vr_lower, vr_upper, alpha=0.25, color='#1565C0',
                label='IC 95% (Bootstrap)')
ax.plot(T_range, vr_median, '-', color='#1565C0', linewidth=2.5,
        label=f'{best_loi} (médiane)')
ax.scatter(T_obs, data_sorted, zorder=10, color='black', s=40,
           label='Maxima observés')
ax.set_xscale('log')
ax.set_xlabel('Période de retour T (ans)')
ax.set_ylabel('Intensité max (mm/h)')
ax.set_title(
    f'Valeurs de retour avec IC 95% — {best_loi} (Bootstrap, n={N_BOOTSTRAP})',
    fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig4_intervalles_confiance.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig4 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 5 — Test de stationnarité (maxima annuels + tendance)
# ==============================================================================
fig, ax = plt.subplots(figsize=(13, 5))

colors_src = {'observé': '#1565C0', 'reconstitué': '#E53935'}
for src, grp in maxima.groupby('source'):
    ax.scatter(grp['annee'], grp['imax_mm_h'], s=60, zorder=5,
               color=colors_src.get(src, 'grey'), label=src.capitalize(), alpha=0.85)

# Droite de régression (tendance linéaire)
x_reg = maxima['annee'].values.astype(float)
y_reg = maxima['imax_mm_h'].values
slope, intercept, r_val, p_val, _ = sp_stats.linregress(x_reg, y_reg)
x_line = np.array([x_reg.min(), x_reg.max()])
ax.plot(x_line, slope * x_line + intercept, '--', color='grey', linewidth=1.5,
        label=f'Tendance linéaire (pente={slope:.2f} mm/h/an, p={p_val:.3f})')

# Annotation du résultat Mann-Kendall
verdict_mk = 'Stationnarité acceptée' if p_mk >= 0.05 else 'Tendance significative'
ax.text(0.02, 0.97,
        f'Mann-Kendall : p = {p_mk:.4f} → {verdict_mk}',
        transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Année')
ax.set_ylabel('Intensité max annuelle (mm/h)')
ax.set_title('Maxima annuels — Vérification de stationnarité',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig5_stationnarite.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig5 sauvegardée')

---
## Résumé

| Indicateur | Valeur |
|------------|--------|
| Source | `data/serie_complete.csv` |
| Années de maxima | 35 (1980–2014) |
| Test de stationnarité | Mann-Kendall (résultat ci-dessus) |
| Loi retenue | Sélectionnée par AIC minimal |
| Tests d'adéquation | KS et Anderson-Darling (résultats ci-dessus) |
| Périodes de retour | 2, 5, 10, 20, 50, 100 ans |
| IC 95% | Bootstrap (n=500) |
| Livrables | 5 figures dans `figures/n6/` |
| Suivant | `n7_courbes_idf.ipynb` |

---
*Fin du Notebook 6*
